# Avance 02 · Kaggle · Etapa 1: delimitación temporal y geográfica
**Equipo A&C | Proyecto: hechos de tránsito en las alcaldías de la CDMX**

## Nuestro objetivo
En esta primera etapa **revisamos los datos originales de Kaggle** y construimos una copia delimitada a **2018–2022** y a las **16 alcaldías de la Ciudad de México**. No sobrescribimos ni alteramos los archivos de origen.

**Advertencia metodológica:** esta fuente contiene una copia histórica de ATUS. **No sumamos** sus accidentes a los de INEGI–ATUS como si fueran observaciones independientes: puede existir solapamiento entre ambas fuentes.

**Contrato de lectura:** trabajamos solo con los archivos `atus_anual_2018.csv` a `atus_anual_2022.csv` dentro de `Kaggle/`. No leemos el archivo `accidentes.db` ni los CSV de la carpeta `INEGI/`.

**Corrección del catálogo:** en esta fuente `001` corresponde a *Ciudad de México* (no a una alcaldía); las 16 alcaldías están identificadas por las claves `002` a `017`, incluida Venustiano Carranza (`017`). **Corrección de años:** normalizamos tabulaciones y espacios en una columna derivada sin alterar `ANIO`. Esta versión reemplaza las salidas previas de Etapa 1 al ejecutarse de principio a fin.

## 1. Importamos bibliotecas y localizamos nuestros archivos
Buscamos `CARPETA_DATOS.zip` o su carpeta extraída en el directorio de trabajo y en sus carpetas superiores. Si existen varias ubicaciones, establecemos una ruta manual para evitar escoger una equivocada.

**Corrección importante:** las filas CSV de Kaggle tienen una coma final adicional. Al usar `index_col=False` impedimos que pandas interprete la primera columna como índice y desplace los valores.

In [1]:
from pathlib import Path
from collections import Counter
from contextlib import contextmanager
import re
import zipfile
import pandas as pd
from IPython.display import display

# Si la detección automática no encuentra tu archivo, escribe aquí su ruta completa.
RUTA_MANUAL = None  # Ejemplo: Path('/home/usuario/proyecto/CARPETA_DATOS.zip')
ANIO_INICIAL, ANIO_FINAL = 2018, 2022
TAM_BLOQUE = 50000

if RUTA_MANUAL is not None:
    ORIGEN = Path(RUTA_MANUAL).expanduser().resolve()
else:
    candidatos = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        for nombre in ['CARPETA_DATOS.zip', 'CARPETA_DATOS']:
            ruta = base / nombre
            if ruta.exists():
                candidatos.append(ruta.resolve())
    candidatos = list(dict.fromkeys(candidatos))
    if len(candidatos) != 1:
        print('Ubicaciones encontradas:', *candidatos, sep='\n - ')
        raise RuntimeError('Especificamos una ruta única en RUTA_MANUAL.')
    ORIGEN = candidatos[0]

ES_ZIP = ORIGEN.is_file() and ORIGEN.suffix.lower() == '.zip'
if not ES_ZIP and not ORIGEN.is_dir():
    raise ValueError('El origen debe ser un ZIP o una carpeta extraída.')

z = zipfile.ZipFile(ORIGEN) if ES_ZIP else None
rutas = z.namelist() if ES_ZIP else [str(p.relative_to(ORIGEN)).replace('\\', '/') for p in ORIGEN.rglob('*') if p.is_file()]

# Solo aceptamos archivos dentro de la fuente Kaggle, no dentro de INEGI.
rutas_kaggle = [r for r in rutas if 'kaggle' in [parte.lower() for parte in Path(r).parts]]
archivos = {}
for ruta in rutas_kaggle:
    m = re.search(r'atus_anual_(\d{4})\.csv$', ruta, re.IGNORECASE)
    if m and ANIO_INICIAL <= int(m.group(1)) <= ANIO_FINAL:
        anio = int(m.group(1))
        if anio in archivos:
            raise RuntimeError(f'Encontramos dos CSV para el año {anio}; revisar rutas.')
        archivos[anio] = ruta

esperados = set(range(ANIO_INICIAL, ANIO_FINAL + 1))
if set(archivos) != esperados:
    raise FileNotFoundError(f'Faltan años: {sorted(esperados - set(archivos))}. Encontrados: {sorted(archivos)}')

catalogos = [r for r in rutas_kaggle if Path(r).name.lower() == 'tc_municipio.csv']
if len(catalogos) != 1:
    raise FileNotFoundError(f'Se esperaba un catálogo municipal dentro de Kaggle; encontramos {len(catalogos)}.')

@contextmanager
def abrir(ruta):
    archivo = z.open(ruta) if ES_ZIP else open(ORIGEN / ruta, 'rb')
    try:
        yield archivo
    finally:
        archivo.close()

SALIDA = ORIGEN.parent / 'resultados' / 'resultados_Kaggle' / 'etapa1'
SALIDA.mkdir(parents=True, exist_ok=True)
APTO = SALIDA / 'Kaggle_ATUS_etapa1.csv'
EXCLUIDO = SALIDA / 'Kaggle_ATUS_excluidos_etapa1.csv'
print('Origen:', ORIGEN)
print('Años encontrados:', sorted(archivos))
print('Catálogo:', catalogos[0])
print('Carpeta de salida:', SALIDA)

Origen: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/CARPETA_DATOS.zip
Años encontrados: [2018, 2019, 2020, 2021, 2022]
Catálogo: CARPETA_DATOS/Kaggle/archive/datos/datos/catalogos/catalogos/tc_municipio.csv
Carpeta de salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa1


## 2. Revisamos el catálogo municipal y elegimos las 16 alcaldías reales

En el catálogo de Kaggle, **`001` no es una alcaldía**: aparece como «Ciudad de México». Las alcaldías corresponden a las claves **`002` a `017`**, incluida **Venustiano Carranza (`017`)**. Por ello corregimos la selección anterior (`001`–`016`).

Primero verificamos los nombres del catálogo. Después construimos nuestro diccionario de correspondencias. Conservamos intacto el catálogo original y guardamos una copia de las 16 alcaldías para la evidencia.

In [2]:
with abrir(catalogos[0]) as f:
    catalogo_original = pd.read_csv(f, dtype=str, encoding='utf-8-sig', index_col=False)

catalogo = catalogo_original.copy()
for col in ['ID_ENTIDAD', 'ID_MUNICIPIO', 'NOM_MUNICIPIO']:
    if col not in catalogo.columns:
        raise KeyError(f'No encontramos {col} en el catálogo: {catalogo.columns.tolist()}')

catalogo['entidad_norm'] = catalogo['ID_ENTIDAD'].str.strip().str.zfill(2)
catalogo['municipio_norm'] = catalogo['ID_MUNICIPIO'].str.strip().str.zfill(3)
alcaldias_catalogo = catalogo.loc[
    catalogo['entidad_norm'].eq('09') & catalogo['municipio_norm'].isin([f'{i:03d}' for i in range(2, 18)]),
    ['entidad_norm', 'municipio_norm', 'NOM_MUNICIPIO']
].drop_duplicates(subset=['entidad_norm', 'municipio_norm']).copy()

if len(alcaldias_catalogo) != 16:
    raise AssertionError(f'Esperábamos 16 alcaldías y encontramos {len(alcaldias_catalogo)}.')

mapa_alcaldias = dict(zip(alcaldias_catalogo['municipio_norm'], alcaldias_catalogo['NOM_MUNICIPIO']))
display(alcaldias_catalogo)
alcaldias_catalogo.to_csv(SALIDA / 'catalogo_alcaldias_CDMX_Kaggle.csv', index=False, encoding='utf-8-sig')
# Comprobamos explícitamente las dos claves que originaron el problema.
assert mapa_alcaldias.get('017', '').strip().casefold() == 'venustiano carranza', 'Revisar clave 017.'
assert '001' not in mapa_alcaldias, 'La clave 001 no debe estar en las 16 alcaldías.'
print('Claves válidas:', sorted(mapa_alcaldias))
print('Venustiano Carranza:', mapa_alcaldias['017'])

,entidad_norm,municipio_norm,NOM_MUNICIPIO
282,09,002,Azcapotzalco
283,09,003,Coyoacán
284,09,004,Cuajimalpa de Morelos
285,09,005,Gustavo A. Madero
286,09,006,Iztacalco
287,09,007,Iztapalapa
288,09,008,La Magdalena Contreras
289,09,009,Milpa Alta
290,09,010,Álvaro Obregón
291,09,011,Tláhuac


Claves válidas: ['002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017']
Venustiano Carranza: Venustiano Carranza


## 3. Delimitamos los archivos anuales por bloques y documentamos cada decisión

Leemos cada CSV en bloques de 50,000 filas para no cargar todo el conjunto nacional en memoria. Como las filas de Kaggle contienen una coma adicional al final, mantenemos `index_col=False`.

**Creamos una copia de cada bloque** antes de transformar. La columna original `ANIO` se conserva exactamente como llegó, incluso cuando contiene `\t2021` o `\t2022`. Creamos `anio_normalizado` para eliminar espacios y tabulaciones y convertir el valor a número. De igual manera, agregamos `entidad_normalizada` y `municipio_normalizado` para validar las claves.

La regla de aceptación requiere: año entre 2018 y 2022 coincidente con el archivo; entidad `09`; y municipio en `002`–`017`. **No exigimos que cada alcaldía tenga accidentes**: comprobamos que todas las claves presentes sean válidas y mostramos cuáles están ausentes, si las hubiera.

Separamos las filas aptas y excluidas en archivos diferentes, sin modificar el ZIP. Documentamos el primer motivo de exclusión por fila y el archivo y número de fila de procedencia. Volver a ejecutar esta celda sobrescribe solamente las salidas derivadas de esta etapa.

In [3]:
inventario = []
por_motivo = Counter()
columnas_originales = None
escribir_aptos = True
escribir_excluidos = True

# Volver a ejecutar esta celda reemplaza SOLO las salidas de la Etapa 1,
# nunca los CSV originales dentro del ZIP o carpeta de Kaggle.
for anio_archivo, ruta in sorted(archivos.items()):
    recibidas = aptas = excluidas = 0
    with abrir(ruta) as f:
        lector = pd.read_csv(f, dtype=str, encoding='utf-8-sig', index_col=False,
                             chunksize=TAM_BLOQUE, low_memory=False)
        for bloque in lector:
            # Trabajamos siempre sobre una copia independiente del bloque leído.
            datos = bloque.copy()
            if columnas_originales is None:
                columnas_originales = list(datos.columns)
            elif list(datos.columns) != columnas_originales:
                raise ValueError(f'Columnas distintas en {ruta}. Revisamos antes de combinar.')
            for col in ['ID_ENTIDAD', 'ID_MUNICIPIO', 'ANIO']:
                if col not in datos.columns:
                    raise KeyError(f'Falta la columna {col} en {ruta}')

            datos['archivo_origen'] = Path(ruta).name
            datos['fila_origen'] = range(recibidas + 2, recibidas + len(datos) + 2)
            entidad = datos['ID_ENTIDAD'].astype('string').str.strip().str.zfill(2)
            municipio = datos['ID_MUNICIPIO'].astype('string').str.strip().str.zfill(3)
            # Normalizamos el año en una columna NUEVA para no modificar ANIO.
            datos['anio_normalizado'] = pd.to_numeric(
                datos['ANIO'].astype('string').str.strip(), errors='coerce'
            ).astype('Int64')
            datos['entidad_normalizada'] = entidad
            datos['municipio_normalizado'] = municipio
            anio = datos['anio_normalizado']

            anio_ok = anio.between(ANIO_INICIAL, ANIO_FINAL) & anio.eq(anio_archivo)
            entidad_ok = entidad.eq('09')
            municipio_ok = municipio.isin(mapa_alcaldias)
            valido = (anio_ok & entidad_ok & municipio_ok).fillna(False)

            # Registramos un solo motivo principal de exclusión por fila.
            # Así los conteos por motivo suman el total de excluidos.
            datos['motivo_exclusion'] = 'apto'
            datos.loc[~anio_ok.fillna(False), 'motivo_exclusion'] = 'anio_invalido_o_no_coincide_archivo'
            datos.loc[anio_ok.fillna(False) & ~entidad_ok.fillna(False), 'motivo_exclusion'] = 'fuera_CDMX'
            datos.loc[anio_ok.fillna(False) & entidad_ok.fillna(False) & ~municipio_ok, 'motivo_exclusion'] = 'municipio_fuera_16_alcaldias'
            datos['alcaldia_normalizada'] = municipio.map(mapa_alcaldias)

            copia_apta = datos.loc[valido].copy()
            copia_excluida = datos.loc[~valido].copy()
            copia_apta.to_csv(APTO, mode='w' if escribir_aptos else 'a',
                              header=escribir_aptos, index=False, encoding='utf-8-sig')
            copia_excluida.to_csv(EXCLUIDO, mode='w' if escribir_excluidos else 'a',
                                  header=escribir_excluidos, index=False, encoding='utf-8-sig')
            escribir_aptos = False
            escribir_excluidos = False
            recibidas += len(datos)
            aptas += len(copia_apta)
            excluidas += len(copia_excluida)
            por_motivo.update(copia_excluida['motivo_exclusion'].value_counts().to_dict())
    inventario.append({'archivo': Path(ruta).name, 'anio_archivo': anio_archivo,
                       'filas_recibidas': recibidas, 'filas_aptas': aptas,
                       'filas_excluidas': excluidas, 'columnas_originales': len(columnas_originales)})
    print(f'{anio_archivo}: recibidas={recibidas:,}; aptas={aptas:,}; excluidas={excluidas:,}')

inventario_df = pd.DataFrame(inventario)
resumen_df = pd.DataFrame([{
    'filas_recibidas': int(inventario_df['filas_recibidas'].sum()),
    'filas_aptas': int(inventario_df['filas_aptas'].sum()),
    'filas_excluidas': int(inventario_df['filas_excluidas'].sum()),
    'archivos_procesados': len(inventario_df)
}])
print('\nInventario:')
display(inventario_df)
print('Resumen:')
display(resumen_df)
inventario_df.to_csv(SALIDA / 'inventario_crudos_Kaggle.csv', index=False, encoding='utf-8-sig')
resumen_df.to_csv(SALIDA / 'resumen_delimitacion_Kaggle.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(sorted(por_motivo.items()), columns=['motivo', 'filas']).to_csv(
    SALIDA / 'frecuencias_motivos_exclusion_Kaggle.csv', index=False, encoding='utf-8-sig')

2018: recibidas=381,708; aptas=11,656; excluidas=370,052
2019: recibidas=378,814; aptas=10,673; excluidas=368,141
2020: recibidas=318,046; aptas=6,549; excluidas=311,497
2021: recibidas=356,315; aptas=6,743; excluidas=349,572
2022: recibidas=392,710; aptas=7,596; excluidas=385,114

Inventario:


,archivo,anio_archivo,filas_recibidas,filas_aptas,filas_excluidas,columnas_originales
0,atus_anual_2018.csv,2018,381708,11656,370052,45
1,atus_anual_2019.csv,2019,378814,10673,368141,45
2,atus_anual_2020.csv,2020,318046,6549,311497,45
3,atus_anual_2021.csv,2021,356315,6743,349572,45
4,atus_anual_2022.csv,2022,392710,7596,385114,45


Resumen:


,filas_recibidas,filas_aptas,filas_excluidas,archivos_procesados
0,1827593,43217,1784376,5


## 4. Verificamos el CSV exportado, sin volver a leer los originales

Leemos la salida apta por bloques y construimos las frecuencias a partir de `anio_normalizado`, no de la columna `ANIO` original. Así evitamos que las tabulaciones de 2021 y 2022 generen categorías separadas.

También revisamos las claves de municipio, calculamos cuántas alcaldías están representadas y cuáles faltan, sin confundir «16 alcaldías permitidas» con «16 alcaldías necesariamente presentes». Guardamos las frecuencias como evidencia reproducible.

In [4]:
por_anio = Counter()
por_alcaldia = Counter()
filas_verificadas = 0
entidades_detectadas = set()
municipios_detectados = set()

for bloque in pd.read_csv(APTO, dtype=str, encoding='utf-8-sig', chunksize=TAM_BLOQUE, index_col=False):
    copia = bloque.copy()
    filas_verificadas += len(copia)
    # Usamos el año normalizado: '2021' y '\t2021' son el mismo año.
    por_anio.update(copia['anio_normalizado'].astype('string').str.strip().value_counts().to_dict())
    por_alcaldia.update(copia['alcaldia_normalizada'].value_counts().to_dict())
    entidades_detectadas.update(copia['entidad_normalizada'].astype('string').str.strip().str.zfill(2).dropna().unique())
    municipios_detectados.update(copia['municipio_normalizado'].astype('string').str.strip().str.zfill(3).dropna().unique())

anios_df = pd.DataFrame(sorted(por_anio.items()), columns=['anio', 'filas'])
alcaldias_df = pd.DataFrame(sorted(por_alcaldia.items()), columns=['alcaldia', 'filas'])
display(anios_df)
display(alcaldias_df)
anios_df.to_csv(SALIDA / 'frecuencias_anios_Kaggle.csv', index=False, encoding='utf-8-sig')
alcaldias_df.to_csv(SALIDA / 'frecuencias_alcaldias_Kaggle.csv', index=False, encoding='utf-8-sig')
print('Filas verificadas:', filas_verificadas)
# Identificamos alcaldías que no aparecen en los registros seleccionados.
alcaldias_presentes = set(por_alcaldia)
alcaldias_ausentes = set(mapa_alcaldias.values()) - alcaldias_presentes
print('Alcaldías presentes:', len(alcaldias_presentes))
print('Alcaldías ausentes:', sorted(alcaldias_ausentes))
print('Clave 017 presente:', '017' in municipios_detectados)


,anio,filas
0,2018,11656
1,2019,10673
2,2020,6549
3,2021,6743
4,2022,7596


,alcaldia,filas
0,Azcapotzalco,1850
1,Benito Juárez,3285
2,Coyoacán,3081
3,Cuajimalpa de Morelos,1035
4,Cuauhtémoc,6317
5,Gustavo A. Madero,5271
6,Iztacalco,1726
7,Iztapalapa,5040
8,La Magdalena Contreras,592
9,Miguel Hidalgo,3892


Filas verificadas: 43217
Alcaldías presentes: 16
Alcaldías ausentes: []
Clave 017 presente: True


## 5. Ejecutamos las comprobaciones de consistencia

Revisamos que los registros recibidos sean iguales a la suma de aptos y excluidos, que las frecuencias correspondan al CSV exportado y que los años y claves geográficas sean válidos.

**Corrección de la regla geográfica:** «solo_16_alcaldias» significa que los municipios presentes son un **subconjunto** de las 16 claves oficiales (`002`–`017`); no exigimos 16 valores distintos en cada periodo. Informamos aparte cuántas alcaldías aparecen realmente.

**Si alguna comprobación falla, no la cambiamos a `True` artificialmente**: mostramos el detalle para investigar. La ausencia de errores en estas pruebas no sustituye la revisión de calidad de la Etapa 2.

In [5]:
recibidas_total = int(resumen_df.loc[0, 'filas_recibidas'])
aptas_total = int(resumen_df.loc[0, 'filas_aptas'])
excluidas_total = int(resumen_df.loc[0, 'filas_excluidas'])
comprobaciones = {
    'recibidas_igual_aptas_mas_excluidas': recibidas_total == aptas_total + excluidas_total,
    'conteo_archivos_2018_2022': len(inventario_df) == 5,
    'frecuencias_anuales_coinciden': sum(por_anio.values()) == aptas_total,
    'frecuencias_alcaldias_coinciden': sum(por_alcaldia.values()) == aptas_total,
    'solo_anios_2018_2022': set(por_anio).issubset({str(a) for a in range(2018, 2023)}),
    'solo_entidad_09': entidades_detectadas == {'09'},
    'solo_16_alcaldias': municipios_detectados.issubset(set(mapa_alcaldias)) and alcaldias_presentes.issubset(set(mapa_alcaldias.values())),
    'salida_igual_aptas': filas_verificadas == aptas_total,
    'originales_disponibles': ORIGEN.exists(),
}
comprobaciones_df = pd.DataFrame(list(comprobaciones.items()), columns=['comprobacion', 'cumple'])
display(comprobaciones_df)
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_Kaggle_etapa1.csv', index=False, encoding='utf-8-sig')
if not all(comprobaciones.values()):
    print('Comprobaciones pendientes:')
    display(comprobaciones_df.loc[~comprobaciones_df['cumple']].copy())
    raise AssertionError('Hay comprobaciones pendientes; revisar antes de continuar.')
print('Etapa 1 Kaggle completada: todas las comprobaciones superadas.')
# Comprobaciones específicas de las correcciones solicitadas.
print('¿Clave 001 incluida en la tabla apta?:', '001' in municipios_detectados)
print('¿Clave 017 incluida en el catálogo?:', '017' in mapa_alcaldias)
print('Años detectados:', sorted(por_anio))
print('Alcaldías presentes:', len(alcaldias_presentes))

,comprobacion,cumple
0,recibidas_igual_aptas_mas_excluidas,True
1,conteo_archivos_2018_2022,True
2,frecuencias_anuales_coinciden,True
3,frecuencias_alcaldias_coinciden,True
4,solo_anios_2018_2022,True
5,solo_entidad_09,True
6,solo_16_alcaldias,True
7,salida_igual_aptas,True
8,originales_disponibles,True


Etapa 1 Kaggle completada: todas las comprobaciones superadas.
¿Clave 001 incluida en la tabla apta?: False
¿Clave 017 incluida en el catálogo?: True
Años detectados: ['2018', '2019', '2020', '2021', '2022']
Alcaldías presentes: 16


In [6]:
print("INVENTARIO DEFINITIVO DE KAGGLE")
print(inventario_df.to_string(index=False))

print("\nRESUMEN DEFINITIVO DE KAGGLE")
print(resumen_df.to_string(index=False))

INVENTARIO DEFINITIVO DE KAGGLE
            archivo  anio_archivo  filas_recibidas  filas_aptas  filas_excluidas  columnas_originales
atus_anual_2018.csv          2018           381708        11656           370052                   45
atus_anual_2019.csv          2019           378814        10673           368141                   45
atus_anual_2020.csv          2020           318046         6549           311497                   45
atus_anual_2021.csv          2021           356315         6743           349572                   45
atus_anual_2022.csv          2022           392710         7596           385114                   45

RESUMEN DEFINITIVO DE KAGGLE
 filas_recibidas  filas_aptas  filas_excluidas  archivos_procesados
         1827593        43217          1784376                    5


## 6. Interpretamos y conservamos las evidencias

**Qué corregimos:** usamos las claves municipales reales `002`–`017`, de modo que `001` («Ciudad de México») no se considera una alcaldía y `017` (Venustiano Carranza) sí puede incorporarse. Normalizamos tabulaciones y espacios del año en una columna derivada, sin modificar el valor original de `ANIO`.

**Archivos de salida:** `Kaggle_ATUS_etapa1.csv` contiene la copia apta; `Kaggle_ATUS_excluidos_etapa1.csv` documenta lo excluido; el inventario, resumen, catálogo, frecuencias y comprobaciones explican el resultado. Estos archivos se regeneran al ejecutar el notebook completo.

**Qué no hacemos:** no deduplicamos ni corregimos víctimas o fechas en esta etapa; eso se analizará en las siguientes. No sumamos los registros de Kaggle con INEGI–ATUS porque pueden proceder de la misma operación estadística.

**Para la bitácora:** registramos los nuevos conteos de recibidos, aptos y excluidos, las 16 claves permitidas, la lista de alcaldías realmente presentes, las comprobaciones y el motivo de las correcciones. **El total previo de 40,293 aptos debe recalcularse**, no conservarse como cifra definitiva.